In [ ]:
import os
os.environ["HF_TOKEN"] = ""

In [ ]:
# @title 1. Install Dependencies
# Install lazyslide and any other requirements (e.g. openslide)
!pip install lazyslide
!apt-get install python3-openslide  # Often needed for WSI handling

In [ ]:
# @title 2. Load Slide and Initialize Storage
import lazyslide as zs
import os

# Define paths
slide_path = "/content/53-adenokarcinom-plic.svs"
zarr_path = "results/53-adenokarcinom-plic.zarr"

# Check if we are resuming a previous run
if os.path.exists(zarr_path):
    print(f"Resuming analysis from {zarr_path}")
else:
    print(f"Starting new analysis for {slide_path}")

# Open the WSI
# If the .zarr file exists, it loads the data from there.
# If not, it prepares a new backing store.
slide = zs.open_wsi(
    slide_path,
    backed_file=zarr_path
)

print(f"Slide loaded: {slide}")

Starting new analysis for /content/53-adenokarcinom-plic.svs
Slide loaded: WSI: /content/53-adenokarcinom-plic.svs
Reader: openslide
Dimensions: 27911×27888 (h×w), 3 Pyramids
Pixel physical size: 0.5009 MPP
SpatialData object
└── Images
      └── 'wsi_thumbnail': DataArray[cyx] (3, 1998, 1997)
with coordinate systems:
    ▸ 'global', with elements:
        wsi_thumbnail (Images)


In [ ]:
# @title 3. Tissue Detection and Tiling
# Check if 'tiles' already exist in the slide object
if "tiles" not in slide.shapes:
    print("Detecting tissue and tiling...")
    zs.pp.find_tissues(slide)

    # FIX: Changed mpp=0.5 to mpp=None to use the slide's native resolution (0.5009)
    zs.pp.tile_tissues(slide, tile_px=256, mpp=None, key_added="tiles")
else:
    print("Tiles found in storage. Skipping preprocessing.")
    print(f"Number of tiles: {len(slide.shapes['tiles'])}")

Tiles found in storage. Skipping preprocessing.
Number of tiles: 4493


In [ ]:
!pip install -q git+https://github.com/mahmoodlab/CONCH.git

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 5.3 MB/s eta 0:00:00


In [ ]:
# @title 4. Feature Extraction
# Define the keys we want to use
model_name = "conch"
feature_key = "conch_feats"

# Retrieve the Hugging Face token from environment variables
hf_token = os.environ.get("HF_TOKEN")

# Check if these features already exist in the slide's tables
if feature_key in slide.tables:
    print(f"Features '{feature_key}' already found! Skipping extraction.")
else:
    print(f"Extracting features using {model_name}...")
    # This might take a few minutes for 4500 tiles
    zs.tl.feature_extraction(
        slide,
        model=model_name,
        tile_key="tiles",
        key_added=feature_key,
        token=hf_token  # Explicitly pass the token here
    )
    print("Extraction complete.")

Extracting features using conch...


/usr/local/lib/python3.12/dist-packages/lazyslide/models/multimodal/conch.py:27: UserWarning: As from v0.8.2, Normalization will not be applied to image embedding of CONCH model anymore.A `normalize=True` argument is added to the `text_image_similarity` method.If you only use the image embedding for text image similarity, you can safely ignore this warning.
  warnings.warn(


meta.yaml:   0%|          | 0.00/37.0 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/802M [00:00<?, ?B/s]

Output()

Extraction complete.


In [ ]:
# @title 5. Define Text Prompts
# Tailored for Lung Adenocarcinoma (LUAD)
prompts = [
    "lung adenocarcinoma",    # The malignant glandular structures
    "desmoplastic stroma",    # The fibrous tissue response often seen in LUAD
    "normal alveoli",         # Healthy lung air sacs (background)
    "lymphocytic infiltrate", # Immune cells fighting the tumor
    "blood vessel",           # Vasculature
    "red blood cells"         # Often helpful to separate hemorrhage from tissue
]

print(f"Encoding prompts: {prompts}")

# Convert text to embeddings (using CONCH as before)
text_embs = zs.tl.text_embedding(prompts, model="conch")
print("Text embeddings ready.")

Encoding prompts: ['lung adenocarcinoma', 'desmoplastic stroma', 'normal alveoli', 'lymphocytic infiltrate', 'blood vessel', 'red blood cells']


/usr/local/lib/python3.12/dist-packages/lazyslide/models/multimodal/conch.py:27: UserWarning: As from v0.8.2, Normalization will not be applied to image embedding of CONCH model anymore.A `normalize=True` argument is added to the `text_image_similarity` method.If you only use the image embedding for text image similarity, you can safely ignore this warning.
  warnings.warn(


Text embeddings ready.


In [ ]:
# @title 6. Compute Text-Image Similarity (Force Update)
similarity_key = "similarity"

print("Calculating new similarity map for updated prompts...")
zs.tl.text_image_similarity(
    slide,
    text_embs,
    feature_key="conch_feats",
    key_added=similarity_key
)
print("Similarity calculation done.")

Calculating new similarity map for updated prompts...
Similarity calculation done.


/usr/local/lib/python3.12/dist-packages/lazyslide/tools/_text_annotate.py:174: UserWarning: As of v0.8.2, the image embedding from image text model is not normalized after feature extraction by default. The normalization is applied here (text_image_similarity),if your features are extracted in previous versions, consider setting normalize=False.
  warnings.warn(msg, stacklevel=find_stack_level())


In [ ]:
# @title 7. Fast Alternative: Generate Shapes from Heatmap (Fixed)
import geopandas as gpd
import pandas as pd
from shapely.ops import unary_union
from wsidata.io import add_shapes  # <--- Essential import for fixing the error

# Settings
similarity_key = "similarity"
output_key = "auto_foci"
threshold_percentile = 0.90

# Get the data
sim_data = slide.tables[similarity_key]
tiles_gdf = slide.shapes['tiles']

annotations = []

print("Generating shapes from similarity maps...")

for prompt in prompts:
    # 1. Check if prompt exists
    if prompt not in sim_data.var_names:
        continue

    # 2. Extract scores
    scores = sim_data[:, prompt].X.flatten()

    # 3. Adaptive Thresholding
    thresh = pd.Series(scores).quantile(threshold_percentile)

    # 4. Select tiles
    mask = scores > thresh
    selected_tiles = tiles_gdf[mask]

    if len(selected_tiles) == 0:
        continue

    print(f"  - '{prompt}': found {len(selected_tiles)} tiles (thresh={thresh:.3f})")

    # 5. Merge tiles
    try:
        merged_shape = unary_union(selected_tiles.geometry).buffer(10).buffer(-10)
    except Exception as e:
        print(f"    Error merging shapes for {prompt}: {e}")
        continue

    if merged_shape.geom_type == 'MultiPolygon':
        shapes = list(merged_shape.geoms)
    else:
        shapes = [merged_shape]

    # 6. Add to list
    for geom in shapes:
        if geom.area < 5000:
            continue
        annotations.append({
            "geometry": geom,
            "classification": prompt
        })

if annotations:
    # Create GeoDataFrame
    gdf = gpd.GeoDataFrame(annotations)

    # FIX: Use add_shapes helper instead of direct assignment
    add_shapes(slide, output_key, gdf)

    print(f"\nSuccess! Generated {len(gdf)} annotations in '{output_key}'")
else:
    print("No annotations found.")

Generating shapes from similarity maps...
  - 'lung adenocarcinoma': found 450 tiles (thresh=0.232)
  - 'desmoplastic stroma': found 450 tiles (thresh=-0.039)
  - 'normal alveoli': found 450 tiles (thresh=0.629)
  - 'lymphocytic infiltrate': found 450 tiles (thresh=-0.027)
  - 'blood vessel': found 450 tiles (thresh=0.204)
  - 'red blood cells': found 450 tiles (thresh=0.270)

Success! Generated 1112 annotations in 'auto_foci'


In [ ]:
# @title 8. Export to QuPath
import os

output_dir = "results"
os.makedirs(output_dir, exist_ok=True)
output_file = f"{output_dir}/adenokarcinom-plic_Suggestions_Fast.geojson"

zs.io.export_annotations(
    slide,
    key="auto_foci",
    file=output_file,
    format="qupath"
)

print(f"POC Saved: {output_file}")

POC Saved: results/adenokarcinom-plic_Suggestions_Fast.geojson


/usr/local/lib/python3.12/dist-packages/pyogrio/geopandas.py:710: UserWarning: 'crs' was not provided.  The output dataset will not have projection information defined and may not be usable in other systems.
  write(


In [ ]:
# @title 9. Launch Side-by-Side Gradio Demo (With Legend)
!pip install -q gradio matplotlib geopandas shapely

import gradio as gr
import geopandas as gpd
import matplotlib.pyplot as plt
import io
from PIL import Image
import numpy as np

# 1. Load Data
geojson_path = "/content/results/adenokarcinom-plic_Suggestions_Fast.geojson"
gdf = gpd.read_file(geojson_path)
gdf.crs = None # Prevent projection errors

# 2. Prepare Thumbnail
thumbnail = slide.images["wsi_thumbnail"]
if hasattr(thumbnail, "values"):
    thumbnail = thumbnail.values
if thumbnail.shape[0] == 3:
    thumbnail = thumbnail.transpose(1, 2, 0)
original_pil = Image.fromarray(thumbnail)

# 3. Get Slide Dimensions
minx, miny, maxx, maxy = slide.properties.bounds
original_w = maxx - minx
original_h = maxy - miny
thumb_h, thumb_w, _ = thumbnail.shape
scale_x = thumb_w / original_w
scale_y = thumb_h / original_h

# 4. Define Colors & Visualization
# We define the colors here so we can use them in the plot AND the legend
COLOR_MAP = {
    "lung adenocarcinoma": "red",
    "desmoplastic stroma": "lime",  # Brighter green for better visibility
    "normal alveoli": "blue",
    "lymphocytic infiltrate": "yellow",
    "blood vessel": "orange",
    "red blood cells": "purple"
}

def update_view(selected_layers):
    fig, ax = plt.subplots(figsize=(10, 10))
    ax.imshow(thumbnail, aspect='equal')
    ax.axis("off")

    if selected_layers:
        subset = gdf[gdf["classification"].isin(selected_layers)].copy()
        if not subset.empty:
            subset.geometry = subset.geometry.translate(xoff=-minx, yoff=-miny)
            subset.geometry = subset.geometry.scale(xfact=scale_x, yfact=scale_y, origin=(0,0))

            for cls in selected_layers:
                layer = subset[subset["classification"] == cls]
                if not layer.empty:
                    # Default to white if class not in map
                    c = COLOR_MAP.get(cls, "white")
                    # aspect=1 fixes the crash
                    layer.plot(ax=ax, facecolor="none", edgecolor=c, linewidth=2, aspect=1)

    buf = io.BytesIO()
    plt.savefig(buf, format="png", bbox_inches="tight", pad_inches=0)
    buf.seek(0)
    plt.close(fig)
    return Image.open(buf)

# 5. Build Interface with Legend
available_classes = list(gdf["classification"].unique())

# Helper to create the legend text
def create_legend_markdown():
    md = "### Legend\n"
    # Emoji map to approximate colors
    emoji_map = {
        "red": "🔴", "lime": "🟢", "blue": "🔵",
        "yellow": "🟡", "orange": "🟠", "purple": "🟣"
    }
    for cls in available_classes:
        color_name = COLOR_MAP.get(cls, "white")
        icon = emoji_map.get(color_name, "⚪")
        # Format: 🔴 Lung Adenocarcinoma
        md += f"* {icon} **{cls.title()}**\n"
    return md

with gr.Blocks(title="LungMAP3: Aim 1 Screening Tool") as demo:
    gr.Markdown("# Aim 1: Automated Pathologic Survey")
    gr.Markdown(
        "**Instructions:** Use the checkboxes to toggle AI suggestions. "
        "The legend below indicates the color coding."
    )

    with gr.Row():
        # Column 1: Controls & Legend
        with gr.Column(scale=1):
            # Dynamic Legend
            gr.Markdown(create_legend_markdown())

            check_group = gr.CheckboxGroup(
                choices=available_classes,
                value=[available_classes[0]] if available_classes else [],
                label="Toggle Layers"
            )
            btn = gr.Button("Refresh View", variant="primary")

        # Column 2: Original H&E
        with gr.Column(scale=2):
            gr.Markdown("### Original H&E (SVS)")
            original_output = gr.Image(value=original_pil, label="Raw Image", type="pil", interactive=False)

        # Column 3: AI Suggestions
        with gr.Column(scale=2):
            gr.Markdown("### AI-Annotated Candidates")
            annotated_output = gr.Image(label="LazySlide Suggestions", type="pil")

    # Wire interactions
    btn.click(fn=update_view, inputs=check_group, outputs=annotated_output)
    check_group.change(fn=update_view, inputs=check_group, outputs=annotated_output)

    # Initialize
    demo.load(fn=update_view, inputs=check_group, outputs=annotated_output)

# 6. Launch
demo.launch(share=True, debug=True)

/tmp/ipython-input-3709679906.py:14: DeprecationWarning: Overriding the CRS of a GeoDataFrame that already has CRS. This unsafe behavior will be deprecated in future versions. Use GeoDataFrame.set_crs method instead
  gdf.crs = None # Prevent projection errors


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://c22d612a3578da868d.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://c22d612a3578da868d.gradio.live
